In [ ]:
import json
import urllib.request
from datetime import datetime

from loguru import logger
from plant_care.config import get_env, load_config
from pyspark.sql import SparkSession

In [20]:
spark = SparkSession.builder.getOrCreate()

# Load config
env = get_env()
cfg = load_config("../project_config.yml", env)

CATALOG = cfg.catalog
SCHEMA = cfg.schema
TABLE_NAME = "plant_care"

# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
logger.info(f"Schema {CATALOG}.{SCHEMA} ready")

2026-11-10 22:37:34.631 | INFO     | __main__:<module>:13 - Schema llmops_dev.srinidhi_ilango98 ready


In [14]:
# Fetch plant care data
# Source: PlantSolve API (CC BY 4.0, https://www.plantsolve.com). Requires a backlink when used.
BASE_URL = "https://www.plantsolve.com/api/v1/plants"
TIMEOUT_SECONDS = 30


def _section(data: dict, key: str) -> dict:
    """
    Get a nested section of a profile, defaulting to an empty dict.

    Args:
        data: Dictionary containing the section
        key: Section name

    Returns:
        The section dictionary, or an empty dict if it is missing or null
    """
    return data.get(key) or {}


def _get_json(url: str) -> dict | list:
    """
    Fetch a URL and decode its JSON body.

    Args:
        url: URL of the JSON resource

    Returns:
        Decoded JSON object or array
    """
    with urllib.request.urlopen(url, timeout=TIMEOUT_SECONDS) as response:
        return json.loads(response.read().decode("utf-8"))


def _recommended(parameters: dict, key: str) -> str | None:
    """
    Get the recommended value of a care parameter.

    Args:
        parameters: Parameters section of a plant profile
        key: Parameter name, such as temperature, humidity, or soilPH

    Returns:
        Recommended value, or None if the parameter is missing
    """
    return _section(parameters, key).get("recommended")


def parse_plant_profile(profile: dict) -> dict:
    """
    Flatten a PlantSolve plant profile into structured care fields.

    Args:
        profile: Raw plant profile from the PlantSolve detail endpoint

    Returns:
        Flat dictionary of plant care fields
    """
    care = _section(profile, "care")
    taxonomy = _section(profile, "taxonomy")
    lighting = _section(profile, "lighting")
    parameters = _section(profile, "parameters")
    nutrients = _section(profile, "nutrients")
    growth = _section(profile, "growthCharacteristics")
    pet_safe = _section(profile, "petSafe")

    return {
        "plant_id": profile.get("id"),
        "slug": profile.get("slug"),
        "common_name": profile.get("commonName"),
        "scientific_name": taxonomy.get("sciName"),
        "family": taxonomy.get("family"),
        "category": profile.get("category"),
        "tags": ", ".join(profile.get("taxonomyTags") or []),
        "summary": profile.get("summary"),
        "difficulty": care.get("difficulty"),
        "placement": care.get("placement"),
        "watering": care.get("watering"),
        "soil": care.get("soil"),
        "toxicity": care.get("toxicity"),
        "light_description": lighting.get("description"),
        "sunlight_hours": lighting.get("sunlightHours"),
        "light_intensity": lighting.get("intensity"),
        "temperature_recommended": _recommended(parameters, "temperature"),
        "humidity_recommended": _recommended(parameters, "humidity"),
        "soil_ph_recommended": _recommended(parameters, "soilPH"),
        "fertilizer_frequency": nutrients.get("fertilizerFrequency"),
        "growth_rate": growth.get("growthRate"),
        "mature_height": growth.get("matureHeight"),
        "pet_safe_cat": pet_safe.get("cat"),
        "pet_safe_dog": pet_safe.get("dog"),
        "url": profile.get("canonicalUrl"),
        "published_at": profile.get("publishedAt"),
        "updated_at": profile.get("updatedAt"),
        "ingestion_timestamp": datetime.now().isoformat(),
    }


def fetch_plant_care() -> list[dict]:
    """
    Fetch all plant care profiles from the PlantSolve API.

    Returns:
        List of structured plant care dictionaries from parse_plant_profile
    """
    index = _get_json(f"{BASE_URL}/index.json")
    logger.info(f"Found {len(index)} plants in PlantSolve index")

    plants = []
    for entry in index:
        slug = entry["slug"]
        try:
            profile = _get_json(f"{BASE_URL}/{slug}.json")
        except Exception as e:
            logger.warning(f"Failed to fetch plant '{slug}': {e}")
            continue
        plants.append(parse_plant_profile(profile))

    return plants


logger.info("Fetching plant care data...")
plants = fetch_plant_care()
logger.info(f"Fetched {len(plants)} plant profiles")
if plants:
    logger.info(f"Sample plant: {plants[0]['common_name']} ({plants[0]['scientific_name']})")

2026-11-10 22:16:16.594 | INFO     | __main__:<module>:122 - Fetching plant care data...


2026-11-10 22:16:16.962 | INFO     | __main__:fetch_plant_care:107 - Found 143 plants in PlantSolve index
2026-11-10 22:16:49.836 | INFO     | __main__:<module>:124 - Fetched 143 plant profiles
2026-11-10 22:16:49.837 | INFO     | __main__:<module>:126 - Sample plant: African Violet (Streptocarpus sect. Saintpaulia)


In [ ]:
from pyspark.sql.types import BooleanType, StringType, StructField, StructType
# Create Delta Table in Unity Catalog
# Store the plant care metadata in a Delta table for downstream processing.

NON_NULLABLE_FIELDS = {"plant_id", "slug", "common_name"}
BOOLEAN_FIELDS = {"pet_safe_cat", "pet_safe_dog"}
FIELD_NAMES = [
    "plant_id",
    "slug",
    "common_name",
    "scientific_name",
    "family",
    "category",
    "tags",
    "summary",
    "difficulty",
    "placement",
    "watering",
    "soil",
    "toxicity",
    "light_description",
    "sunlight_hours",
    "light_intensity",
    "temperature_recommended",
    "humidity_recommended",
    "soil_ph_recommended",
    "fertilizer_frequency",
    "growth_rate",
    "mature_height",
    "pet_safe_cat",
    "pet_safe_dog",
    "url",
    "published_at",
    "updated_at",
    "ingestion_timestamp",
]

schema = StructType(
    [
        StructField(
            name,
            BooleanType() if name in BOOLEAN_FIELDS else StringType(),
            name not in NON_NULLABLE_FIELDS,
        )
        for name in FIELD_NAMES
    ]
)

# Create DataFrame
df = spark.createDataFrame(plants, schema=schema)
logger.info(f"Created DataFrame with {df.count()} plant records")

2026-11-10 22:35:16.510 | INFO     | __main__:<module>:48 - Created DataFrame with 143 plant records


In [21]:
# Write to Delta table
table_path = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_path)

logger.info(f"Created Delta table: {table_path}")
logger.info(f"Records: {df.count()}")

2026-11-10 22:39:15.487 | INFO     | __main__:<module>:6 - Created Delta table: llmops_dev.srinidhi_ilango98.plant_care
2026-11-10 22:39:15.973 | INFO     | __main__:<module>:7 - Records: 143


In [22]:
# COMMAND ----------
# Verify the Data

# Read back the table
plants_df = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")

logger.info(f"Table: {CATALOG}.{SCHEMA}.{TABLE_NAME}")
logger.info(f"Total plants: {plants_df.count()}")
logger.info("Schema:")
plants_df.printSchema()

logger.info("Sample records:")
plants_df.select("plant_id", "slug", "common_name", "scientific_name").show(5, truncate=50)

# COMMAND ----------
# Data Statistics

logger.info("Plants by category:")
plants_df.groupBy("category").count().orderBy("count", ascending=False).show()

logger.info("Most recently updated plants:")
plants_df.select("common_name", "updated_at", "plant_id").orderBy("updated_at", ascending=False).show(
    5, truncate=60
)


2026-11-10 22:39:53.521 | INFO     | __main__:<module>:6 - Table: llmops_dev.srinidhi_ilango98.plant_care
2026-11-10 22:39:54.249 | INFO     | __main__:<module>:7 - Total plants: 143
2026-11-10 22:39:54.251 | INFO     | __main__:<module>:8 - Schema:
2026-11-10 22:39:54.642 | INFO     | __main__:<module>:11 - Sample records:


root
 |-- plant_id: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- common_name: string (nullable = true)
 |-- scientific_name: string (nullable = true)
 |-- family: string (nullable = true)
 |-- category: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- difficulty: string (nullable = true)
 |-- placement: string (nullable = true)
 |-- watering: string (nullable = true)
 |-- soil: string (nullable = true)
 |-- toxicity: string (nullable = true)
 |-- light_description: string (nullable = true)
 |-- sunlight_hours: string (nullable = true)
 |-- light_intensity: string (nullable = true)
 |-- temperature_recommended: string (nullable = true)
 |-- humidity_recommended: string (nullable = true)
 |-- soil_ph_recommended: string (nullable = true)
 |-- fertilizer_frequency: string (nullable = true)
 |-- growth_rate: string (nullable = true)
 |-- mature_height: string (nullable = true)
 |-- pet_safe_cat: boolean (nullable 

2026-11-10 22:39:56.201 | INFO     | __main__:<module>:16 - Plants by category:


+---------------------------+---------------------+---------------------+-----------------------------------------------+
|                   plant_id|                 slug|          common_name|                                scientific_name|
+---------------------------+---------------------+---------------------+-----------------------------------------------+
|       plant-african-violet|       african-violet|       African Violet|                Streptocarpus sect. Saintpaulia|
|   plant-aglaonema-red-siam|   aglaonema-red-siam|   Aglaonema Red Siam|                        Aglaonema 'Siam Aurora'|
|   plant-alocasia-amazonica|   alocasia-amazonica|   Alocasia Amazonica|Alocasia x amazonica (commonly sold as 'Polly')|
|plant-alocasia-black-velvet|alocasia-black-velvet|Alocasia Black Velvet|                              Alocasia reginula|
| plant-alocasia-frydek-care|      alocasia-frydek|      Alocasia Frydek|                Alocasia micholitziana 'Frydek'|
+-----------------------

2026-11-10 22:39:56.956 | INFO     | __main__:<module>:19 - Most recently updated plants:


+--------------------+-----+
|            category|count|
+--------------------+-----+
|       Indoor Plants|   92|
|                NULL|   35|
|      Aquatic Plants|    6|
|  Carnivorous Plants|    3|
|    Flowering Plants|    3|
|    Ayurvedic Plants|    1|
|Herbs & Edible Pl...|    1|
|          Succulents|    1|
|Herbs & Outdoor P...|    1|
+--------------------+-----+

+----------------------+----------+---------------------------+
|           common_name|updated_at|                   plant_id|
+----------------------+----------+---------------------------+
|    Mexican Butterwort|2026-10-23|plant-pinguicula-moranensis|
| Sansevieria Moonshine|2026-10-19|plant-sansevieria-moonshine|
|Madagascar Dragon Tree|2026-10-19|   plant-dracaena-marginata|
|            Areca Palm|2026-10-18|           plant-areca-palm|
|          Anubias Nana|2026-10-18|         plant-anubias-nana|
+----------------------+----------+---------------------------+
only showing top 5 rows
